# Our approach — DeepSeek-V3 on αNLI (500 claims, earlier sample from the training split)

Pipeline, run in batches of 100 claims, then results and extension statistics.  
Run on Kaggle; the API key is read from Kaggle secrets (`DEEPSEEK_API_KEY`). Outputs are kept so the reported numbers can be checked.

In [ ]:
!pip install -q openai datasets scikit-learn
import os, re, json, time, pickle, random
from pathlib import Path
from itertools import combinations
import numpy as np, pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from datasets import load_dataset

WORK_DIR = Path("/kaggle/working/anli_ext"); WORK_DIR.mkdir(parents=True, exist_ok=True)
client = OpenAI(api_key=UserSecretsClient().get_secret("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com")
MODEL_NAME = "deepseek-chat"

ds = load_dataset("allenai/art", split="train", trust_remote_code=False)
adf = ds.to_pandas()
random.seed(42)
rows = []
sample = adf.sample(400, random_state=42).reset_index(drop=True)
for _, r in sample.iterrows():
    o1 = str(r["observation_1"]).strip(); o2 = str(r["observation_2"]).strip()
    correct = str(r["hypothesis_1"] if r["label"]==1 else r["hypothesis_2"]).strip()
    wrong   = str(r["hypothesis_2"] if r["label"]==1 else r["hypothesis_1"]).strip()
    rows.append({"claim": f"{o1} {o2} This is because: {correct}", "gold": 1})
    rows.append({"claim": f"{o1} {o2} This is because: {wrong}", "gold": 0})
cdf = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
t = cdf[cdf["gold"]==1].head(250); f = cdf[cdf["gold"]==0].head(250)
baf_df = pd.concat([t,f]).sample(frac=1, random_state=42).reset_index(drop=True)
baf_df["example_index"] = range(len(baf_df))
print("αNLI claims:", len(baf_df), "| True:", baf_df["gold"].sum(), "| first:", baf_df.iloc[0]["claim"][:60])

αNLI claims: 500 | True: 250 | first: My sister in law was a trolley driver. An inspector restarte


In [ ]:
BAF_CONFIG = {
    "n_pro_candidates": 5, "n_con_candidates": 5, "max_pro_keep": 4, "max_con_keep": 4,
    "min_relevance": 0.60, "min_grounding": 0.55, "min_clarity": 0.60, "min_quality": 0.58,
    "min_target_match": 0.65, "min_grounding_edge": 0.55, "min_relation_score": 0.68,
    "max_conflict_pairs": 3, "max_claim_attacks": 2, "audit_batch_size": 16,
    "generation_temperature": 0.15, "audit_temperature": 0.0, "max_retries": 5,
}
def clip01(v):
    try: return max(0.0,min(1.0,float(v)))
    except: return 0.0
def extract_json_object(text):
    text=str(text or "").strip(); text=re.sub(r"```(?:json)?","",text,flags=re.IGNORECASE).strip()
    try: return json.loads(text)
    except json.JSONDecodeError: pass
    s=text.find("{"); e=text.rfind("}")
    if s>=0 and e>s: return json.loads(text[s:e+1])
    raise ValueError("No JSON")
def deepseek_json_call(prompt, temperature, max_tokens, retries=5):
    last=None
    for a in range(1,retries+1):
        try:
            resp=client.chat.completions.create(model=MODEL_NAME,
                messages=[{"role":"system","content":"You are an expert in claim verification, formal argumentation and strict JSON output."},
                          {"role":"user","content":prompt}],
                temperature=temperature,max_tokens=max_tokens)
            t=resp.choices[0].message.content
            if not t: raise ValueError("empty")
            return extract_json_object(t)
        except Exception as e:
            last=e
            if "rate" in str(e).lower() or "429" in str(e): time.sleep(8); continue
            if a<retries: time.sleep(3*a)
    raise RuntimeError(f"failed: {last}")
def generate_facts(claim):
    prompt = f"""You are building atomic factual building blocks.
For the claim below, produce 3-5 SHORT, SEPARATE factual statements. Each must be:
- A single atomic fact about ONE concept in the claim.
- General commonsense knowledge, stated on its own.
- NOT a comparison, ranking, or conclusion. NOT about whether the claim is true or false.
Claim: "{claim}"
Return 3-5 atomic facts, one per line. Do not conclude."""
    for _ in range(5):
        try:
            resp=client.chat.completions.create(model=MODEL_NAME,
                messages=[{"role":"system","content":"You output atomic facts. You never conclude."},
                          {"role":"user","content":prompt}], temperature=0.3, max_tokens=400)
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if "rate" in str(e).lower() or "429" in str(e): time.sleep(8); continue
            return ""
    return ""
print("✅ CELL 2 ready")

✅ CELL 2 ready


In [ ]:
def generate_and_score_arguments(claim, facts_text, config=BAF_CONFIG):
    prompt = f"""
Construct candidate arguments for a Bipolar Argumentation Framework used for claim verification.

CLAIM:
{claim}

AVAILABLE FACTS:
{facts_text}

First determine the intended meaning of the claim, then reason carefully from facts and commonsense.

Generate up to {config["n_pro_candidates"]} PRO candidates and up to {config["n_con_candidates"]} CON candidates.

PRO: a specific, grounded reason the claim is TRUE.
CON: a specific, grounded reason the claim is FALSE.

CRITICAL rules for CON arguments (attacks):
- Only generate a CON if there is a GENUINE, SPECIFIC, STRONG reason the claim is false.
- Do NOT generate speculative CONs like "maybe not always", "it might not", "in some cases".
- Do NOT generate a CON that merely raises a weak possibility; it must give a real reason the claim is false.
- If the claim is clearly true and there is no strong objection, generate ZERO CON arguments.

CRITICAL rules for PRO arguments:
- Only generate a PRO if there is genuine support. If the claim is clearly false, generate ZERO PRO arguments.
- Do NOT generate circular or vague arguments.

General:
- Each argument must be specific and grounded, under 30 words.
- Do not output the final TRUE/FALSE label.

Score each candidate 0 to 1 for: relevance; grounding; clarity; overall_quality.
Score speculative, weak, vague, or circular arguments BELOW 0.4 on overall_quality.

Return valid JSON only:
{{
  "interpreted_claim": "brief meaning",
  "pro": [{{"text":"...","relevance":0.0,"grounding":0.0,"clarity":0.0,"overall_quality":0.0}}],
  "con": [{{"text":"...","relevance":0.0,"grounding":0.0,"clarity":0.0,"overall_quality":0.0}}]
}}
""".strip()
    raw=deepseek_json_call(prompt,config["generation_temperature"],1900,config["max_retries"])
    out={"interpreted_claim":str(raw.get("interpreted_claim","")).strip(),"pro":[],"con":[]}
    for role in ("pro","con"):
        for e in (raw.get(role,[]) if isinstance(raw.get(role,[]),list) else []):
            if not isinstance(e,dict): continue
            t=str(e.get("text","")).strip()
            if not t: continue
            out[role].append({"text":t,"relevance":clip01(e.get("relevance")),"grounding":clip01(e.get("grounding")),"clarity":clip01(e.get("clarity")),"overall_quality":clip01(e.get("overall_quality"))})
    return out

def select_arguments(gen, config=BAF_CONFIG):
    nodes={"CLAIM":{"role":"CLAIM","text":""}}; scored=[]
    for src,role,pre,mx in [("pro","PRO","P",config["max_pro_keep"]),("con","CON","N",config["max_con_keep"])]:
        cands=[]
        for it in gen.get(src,[]):
            rl,g=clip01(it.get("relevance")),clip01(it.get("grounding")); cl,q=clip01(it.get("clarity")),clip01(it.get("overall_quality"))
            rank=0.30*rl+0.30*g+0.15*cl+0.25*q; sc={**it,"role":role,"rank_score":rank}; scored.append(sc)
            if rl>=config["min_relevance"] and g>=config["min_grounding"] and cl>=config["min_clarity"] and q>=config["min_quality"]: cands.append(sc)
        cands.sort(key=lambda x:x["rank_score"],reverse=True)
        for n,it in enumerate(cands[:mx],1):
            nodes[f"{pre}{n}"]={"role":role,"text":it["text"],"relevance":it["relevance"],"grounding":it["grounding"],"clarity":it["clarity"],"overall_quality":it["overall_quality"],"rank_score":it["rank_score"]}
    return {"nodes":nodes,"all_scored":scored}

def build_relation_candidates(nodes):
    pro=[n for n,i in nodes.items() if i["role"]=="PRO"]; con=[n for n,i in nodes.items() if i["role"]=="CON"]; c=[]
    for p in pro: c.append({"source":p,"target":"CLAIM","allowed":["SUPPORT","NONE"]})
    for n in con: c.append({"source":n,"target":"CLAIM","allowed":["ATTACK","NONE"]})
    for p in pro:
        for n in con:
            c.append({"source":p,"target":n,"allowed":["ATTACK","NONE"]}); c.append({"source":n,"target":p,"allowed":["ATTACK","NONE"]})
    return c
print("✅ piece 1 ready")

✅ piece 1 ready


In [ ]:
def audit_baf_relations(claim, facts_text, nodes, config=BAF_CONFIG, batch_size=16):
    cands=build_relation_candidates(nodes)
    at="\n".join(f"{nid} [{i['role']}]: {i['text']}" for nid,i in nodes.items() if nid!="CLAIM")
    aud=[]
    for bs in range(0,len(cands),batch_size):
        b=cands[bs:bs+batch_size]
        pb="\n".join(f"{it['source']} -> {it['target']} | allowed={it['allowed']}" for it in b)
        prompt=f"""Audit candidate relations for a Bipolar Argumentation Framework.

CLAIM:
{claim}
FACTS:
{facts_text}
ARGUMENTS:
{at}

For every candidate, choose exactly one allowed type.
SUPPORT: source gives a meaningful direct reason to accept target.
ATTACK: source directly contradicts, rebuts or undermines target.
NONE: no meaningful direct relation.
Use ATTACK for CON->CLAIM only if the CON gives a genuine reason the claim is false or doubtful.
Use NONE when the CON concedes the claim or agrees.
Keep every reason at eight words or fewer.
For ATTACK/SUPPORT scores normally exceed 0.50; for NONE normally below 0.50.

CANDIDATES:
{pb}

Return valid JSON only:
{{"relations":[{{"source":"P1","target":"CLAIM","type":"SUPPORT","target_match":0.9,"grounding":0.9,"relation_score":0.9,"reason":"..."}}]}}"""
        raw=deepseek_json_call(prompt,config["audit_temperature"],2200,config["max_retries"])
        exp={(it["source"],it["target"]) for it in b}
        allow={(it["source"],it["target"]):set(it["allowed"]) for it in b}
        ret={}
        for r in (raw.get("relations",[]) if isinstance(raw.get("relations",[]),list) else []):
            if not isinstance(r,dict): continue
            s,t=str(r.get("source","")).strip(),str(r.get("target","")).strip()
            typ=str(r.get("type","NONE")).strip().upper(); pr=(s,t)
            if pr not in exp or typ not in allow[pr]: continue
            ret[pr]={"source":s,"target":t,"type":typ,"target_match":clip01(r.get("target_match")),"grounding":clip01(r.get("grounding")),"relation_score":clip01(r.get("relation_score")),"reason":str(r.get("reason","")).strip()}
        for it in b:
            pr=(it["source"],it["target"])
            if pr not in ret:
                ret[pr]={"source":pr[0],"target":pr[1],"type":"NONE","target_match":0.0,"grounding":0.0,"relation_score":0.0,"reason":"omitted"}
        aud.extend(ret.values())
    return {"all_audited_relations":aud}
print("✅ piece 2 ready")

✅ piece 2 ready


In [ ]:
def relation_passes(r,config=BAF_CONFIG): return r["target_match"]>=config["min_target_match"] and r["grounding"]>=config["min_grounding_edge"] and r["relation_score"]>=config["min_relation_score"]
def relation_lookup(rels): return {(r["source"],r["target"]):r for r in rels}
def select_formal_supports(rels,config=BAF_CONFIG): return sorted(set((r["source"],r["target"]) for r in rels if r["type"]=="SUPPORT" and r["target"]=="CLAIM" and r["source"].startswith("P") and relation_passes(r,config)))
def select_con_claim_attacks(rels,config=BAF_CONFIG):
    p=[r for r in rels if r["type"]=="ATTACK" and r["target"]=="CLAIM" and r["source"].startswith("N") and relation_passes(r,config)]
    p.sort(key=lambda x:x["relation_score"],reverse=True); return p[:config["max_claim_attacks"]]
def select_independent_conflict_pairs(rels,config=BAF_CONFIG):
    lk=relation_lookup(rels); cand=[]
    pro=sorted({r["source"] for r in rels if r["source"].startswith("P")}); con=sorted({r["source"] for r in rels if r["source"].startswith("N")})
    for p in pro:
        for n in con:
            pc,cp=lk.get((p,n)),lk.get((n,p))
            if not pc or not cp: continue
            if pc["type"]=="ATTACK" and cp["type"]=="ATTACK" and relation_passes(pc,config) and relation_passes(cp,config):
                cand.append({"pro":p,"con":n,"pair_score":min(pc["relation_score"],cp["relation_score"])})
    cand.sort(key=lambda x:x["pair_score"],reverse=True); sel=[]; up=set(); uc=set()
    for c in cand:
        if c["pro"] in up or c["con"] in uc: continue
        sel.append(c); up.add(c["pro"]); uc.add(c["con"])
        if len(sel)>=config["max_conflict_pairs"]: break
    return sel,cand
def support_closure(nodes,sup):
    cl=set(sup); ch=True
    while ch:
        ch=False; new=set()
        for a,b in cl:
            for c,d in cl:
                if b==c and (a,d) not in cl: new.add((a,d))
        if new: cl|=new; ch=True
    return cl
def derive_attacks(nodes,direct,sup):
    sc=support_closure(nodes,sup); sa=set(); se=set()
    for s,sd in sc:
        for at,t in direct:
            if sd==at: sa.add((s,t))
    for at,ad in direct:
        for s,sd in sc:
            if ad==s: se.add((at,sd))
    eff=(set(direct)|sa|se); eff={e for e in eff if e[0]!=e[1]}
    return {"support_closure":sorted(sc),"supported_attacks":sorted(sa),"secondary_attacks":sorted(se),"effective_attacks":sorted(eff)}
print("✅ piece 3 ready")

✅ piece 3 ready


In [ ]:
def all_subsets(items):
    il=sorted(items)
    for sz in range(len(il)+1):
        for sub in combinations(il,sz): yield set(sub)
def cf(e,att): return not any(s in e and t in e for s,t in att)
def att_of(a,att): return {s for s,t in att if t==a}
def ext_att(e,a,att): return any(s in e and t==a for s,t in att)
def defends(e,a,att): return all(ext_att(e,x,att) for x in att_of(a,att))
def admissible(e,att): return cf(e,att) and all(defends(e,a,att) for a in e)
def preferred_ext(args,att):
    adm=[s for s in all_subsets(args) if admissible(s,att)]
    return [c for c in adm if not any(c<o for o in adm)]
def stable_ext(args,att):
    out=[]
    for c in all_subsets(args):
        if not cf(c,att): continue
        if all(ext_att(c,a,att) for a in args-c): out.append(c)
    return out
def norm(exts): return [sorted(e) for e in exts]
def decisions(exts,cid="CLAIM"):
    if not exts: return {"credulous":0,"skeptical":0,"majority":0}
    cc=sum(cid in e for e in exts)
    return {"credulous":int(cc>=1),"skeptical":int(cc==len(exts)),"majority":int(cc*2>=len(exts))}
def rebuild(inter,config=BAF_CONFIG):
    nodes=dict(inter["nodes"]); rels=inter["all_audited_relations"]; sup=set(select_formal_supports(rels,config))
    sca=select_con_claim_attacks(rels,config); sp,_=select_independent_conflict_pairs(rels,config); direct=set()
    for r in sca: direct.add((r["source"],r["target"]))
    for p in sp: direct.add((p["pro"],p["con"])); direct.add((p["con"],p["pro"]))
    conn={"CLAIM"}
    for st in sup|direct: conn.add(st[0]); conn.add(st[1])
    nodes={k:v for k,v in nodes.items() if k in conn}; vids=set(nodes)
    sup={e for e in sup if e[0] in vids and e[1] in vids}; direct={e for e in direct if e[0] in vids and e[1] in vids}
    der=derive_attacks(vids,direct,sup); eff=set(map(tuple,der["effective_attacks"]))
    pref=preferred_ext(vids,eff); stab=stable_ext(vids,eff)
    return {**inter,"nodes":nodes,"supports":sorted(sup),"selected_claim_attacks":sca,
        "selected_conflict_pairs":[{"pro":p["pro"],"con":p["con"],"pair_score":p["pair_score"]} for p in sp],
        "direct_attacks":sorted(direct),"support_closure":der["support_closure"],"supported_attacks":der["supported_attacks"],
        "secondary_attacks":der["secondary_attacks"],"effective_attacks":der["effective_attacks"],
        "preferred_extensions":norm(pref),"stable_extensions":norm(stab),"preferred_decisions":decisions(pref),"stable_decisions":decisions(stab)}
def run_v3_baf_example(row,config=BAF_CONFIG):
    claim=str(row["claim"]).strip(); facts=str(row["facts_text"]).strip(); gold=int(row["gold"])
    gen=generate_and_score_arguments(claim,facts,config); sel=select_arguments(gen,config); nodes=sel["nodes"]; nodes["CLAIM"]["text"]=claim
    audit=audit_baf_relations(claim,facts,nodes,config,config["audit_batch_size"])
    inter={"example_index":int(row["example_index"]),"claim":claim,"facts_text":facts,"gold":gold,"interpreted_claim":gen["interpreted_claim"],"generated_arguments":gen,"argument_scores":sel["all_scored"],"nodes":nodes,"all_audited_relations":audit["all_audited_relations"],"model":MODEL_NAME,"config":dict(config)}
    return rebuild(inter,config)
print("✅ piece 4 ready — pipeline complete")

✅ piece 4 ready — pipeline complete


In [ ]:
row = baf_df.iloc[0].copy()
row["facts_text"] = generate_facts(row["claim"])
print("CLAIM:", row["claim"])
test = run_v3_baf_example(row)
print(f"gold={test['gold']} pred={test['preferred_decisions']['credulous']}")
print(f"nodes={len(test['nodes'])} supports={len(test['supports'])} attacks={len(test['effective_attacks'])} ext={len(test['preferred_extensions'])}")

CLAIM: My sister in law was a trolley driver. An inspector restarted the trolley. This is because: It broke up when she was driving it.
gold=0 pred=1
nodes=3 supports=2 attacks=0 ext=1


In [ ]:
CKPT = WORK_DIR / "anli_ckpt.pkl"
def load_ck(): return pickle.load(open(CKPT,"rb")) if CKPT.exists() else {}
def save_ck(d): pickle.dump(d, open(CKPT,"wb"))

def run_batch(n=100):
    done = load_ck()
    todo = [i for i in range(len(baf_df)) if i not in done][:n]
    print(f"Have {len(done)}/{len(baf_df)}. Running {len(todo)}...")
    for k, idx in enumerate(todo, 1):
        row = baf_df.iloc[idx].copy()
        try:
            row["facts_text"] = generate_facts(row["claim"])
            res = run_v3_baf_example(row)
        except Exception as e:
            print(f"error at {idx}: {str(e)[:50]}"); continue
        d = load_ck(); d[idx] = res; save_ck(d)
        if k % 10 == 0 or k == len(todo):
            cur = load_ck()
            acc = np.mean([cur[i]["preferred_decisions"]["credulous"]==cur[i]["gold"] for i in cur])
            print(f"  {k}/{len(todo)} → {len(cur)} total | running acc {acc*100:.1f}%")
    print(f"✅ {len(load_ck())}/{len(baf_df)} done")

run_batch(100)

Have 0/500. Running 100...
  10/100 → 10 total | running acc 60.0%
  20/100 → 20 total | running acc 70.0%
  30/100 → 30 total | running acc 63.3%
  40/100 → 40 total | running acc 67.5%
  50/100 → 50 total | running acc 70.0%
  60/100 → 60 total | running acc 73.3%
  70/100 → 70 total | running acc 72.9%
  80/100 → 80 total | running acc 70.0%
  90/100 → 90 total | running acc 70.0%
  100/100 → 100 total | running acc 70.0%
✅ 100/500 done


In [ ]:
run_batch(100)

Have 100/500. Running 100...
  10/100 → 110 total | running acc 68.2%
  20/100 → 120 total | running acc 68.3%
  30/100 → 130 total | running acc 68.5%
  40/100 → 140 total | running acc 67.9%
  50/100 → 150 total | running acc 67.3%
  60/100 → 160 total | running acc 67.5%
  70/100 → 170 total | running acc 67.6%
  80/100 → 180 total | running acc 68.3%
  90/100 → 190 total | running acc 68.4%
  100/100 → 200 total | running acc 68.5%
✅ 200/500 done


In [ ]:
run_batch(100)

Have 200/500. Running 100...
  10/100 → 210 total | running acc 69.5%
  20/100 → 220 total | running acc 68.6%
  30/100 → 230 total | running acc 68.3%
  40/100 → 240 total | running acc 68.8%
  50/100 → 250 total | running acc 69.6%
  60/100 → 260 total | running acc 69.2%
  70/100 → 270 total | running acc 69.6%
  80/100 → 280 total | running acc 68.9%
  90/100 → 290 total | running acc 68.6%
  100/100 → 300 total | running acc 69.0%
✅ 300/500 done


In [ ]:
run_batch(100)

Have 300/500. Running 100...
  10/100 → 310 total | running acc 69.0%
  20/100 → 320 total | running acc 69.1%
  30/100 → 330 total | running acc 69.7%
  40/100 → 340 total | running acc 69.4%
  50/100 → 350 total | running acc 69.4%
  60/100 → 360 total | running acc 69.4%
  70/100 → 370 total | running acc 69.5%
  80/100 → 380 total | running acc 69.5%
  90/100 → 390 total | running acc 69.7%
  100/100 → 400 total | running acc 69.5%
✅ 400/500 done


In [ ]:
run_batch(100)

Have 400/500. Running 100...
  10/100 → 410 total | running acc 69.0%
  20/100 → 420 total | running acc 69.3%
  30/100 → 430 total | running acc 69.3%
  40/100 → 440 total | running acc 69.5%
  50/100 → 450 total | running acc 69.3%
  60/100 → 460 total | running acc 69.6%
  70/100 → 470 total | running acc 69.6%
  80/100 → 480 total | running acc 69.4%
  90/100 → 490 total | running acc 69.2%
  100/100 → 500 total | running acc 69.6%
✅ 500/500 done


In [ ]:
results = [load_ck()[k] for k in sorted(load_ck())]
gold = [r["gold"] for r in results]
n = len(results)

def net_support_pred(r):
    return int(len(r["supports"]) >= sum(1 for (s,t) in r["effective_attacks"] if t=="CLAIM"))
def no_objection_pred(r, exts_key):
    exts = r[exts_key]
    if not exts: return 0
    eff = set(map(tuple, r["effective_attacks"]))
    for ext in exts:
        es = set(ext)
        if "CLAIM" in es and not any(t=="CLAIM" and s in es for (s,t) in eff): return 1
    return 0

def metrics_row(name, pred):
    tn, fp, fn, tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
    return {"method": name,
        "accuracy": round(accuracy_score(gold,pred)*100,1),
        "precision": round(precision_score(gold,pred,zero_division=0)*100,1),
        "recall": round(recall_score(gold,pred,zero_division=0)*100,1),
        "f1": round(f1_score(gold,pred,zero_division=0)*100,1),
        "TN":tn,"FP":fp,"FN":fn,"TP":tp}

rows = []
for rule in ["credulous","skeptical","majority"]:
    rows.append(metrics_row(f"preferred_{rule}", [r["preferred_decisions"][rule] for r in results]))
rows.append(metrics_row("preferred_no_objection", [no_objection_pred(r,"preferred_extensions") for r in results]))
for rule in ["credulous","skeptical","majority"]:
    rows.append(metrics_row(f"stable_{rule}", [r["stable_decisions"][rule] for r in results]))
rows.append(metrics_row("stable_no_objection", [no_objection_pred(r,"stable_extensions") for r in results]))
rows.append(metrics_row("net_support", [net_support_pred(r) for r in results]))

table = pd.DataFrame(rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
pd.set_option("display.max_columns", None); pd.set_option("display.width", 220)
print(f"=== αNLI — ALL semantics ({n} examples) ===\n")
print(table.to_string())
print(f"\ngold balance: True={sum(gold)} False={n-sum(gold)}")
print(f"\nFor comparison:")
print(f"  DF-QuAD 0.5 Base (fair): 54.0%  (at 200)")
print(f"  DF-QuAD Est. Base:       74.0%  (at 200)")
print(f"  just-ask:                76.0%")

=== αNLI — ALL semantics (500 examples) ===

                   method  accuracy  precision  recall    f1   TN   FP  FN   TP
0     preferred_skeptical      71.4       68.5    79.2  73.5  159   91  52  198
1        stable_skeptical      71.4       68.5    79.2  73.5  159   91  52  198
2         stable_majority      70.4       66.7    81.6  73.4  148  102  46  204
3      preferred_majority      70.4       66.7    81.6  73.4  148  102  46  204
4     preferred_credulous      69.6       65.2    84.0  73.4  138  112  40  210
5        stable_credulous      69.6       65.2    84.0  73.4  138  112  40  210
6  preferred_no_objection      69.6       65.2    84.0  73.4  138  112  40  210
7     stable_no_objection      69.6       65.2    84.0  73.4  138  112  40  210
8             net_support      58.4       54.8    96.8  69.9   50  200   8  242

gold balance: True=250 False=250

For comparison:
  DF-QuAD 0.5 Base (fair): 54.0%  (at 200)
  DF-QuAD Est. Base:       74.0%  (at 200)
  just-ask:       

In [ ]:
results = [load_ck()[k] for k in sorted(load_ck())]
n = len(results)
from collections import Counter

print(f"=== αNLI — Extension Statistics ({n} examples) ===\n")

# preferred extensions per example
pref_counts = Counter(len(r["preferred_extensions"]) for r in results)
print("PREFERRED extensions per example:")
for k in sorted(pref_counts):
    print(f"  {k} extension(s): {pref_counts[k]:3d} examples ({pref_counts[k]/n*100:.1f}%)")
print(f"  → avg: {np.mean([len(r['preferred_extensions']) for r in results]):.2f} | max: {max(len(r['preferred_extensions']) for r in results)}")

# stable extensions
stab_counts = Counter(len(r["stable_extensions"]) for r in results)
print("\nSTABLE extensions per example:")
for k in sorted(stab_counts):
    print(f"  {k} extension(s): {stab_counts[k]:3d} examples ({stab_counts[k]/n*100:.1f}%)")
print(f"  → avg: {np.mean([len(r['stable_extensions']) for r in results]):.2f}")

# conflict pairs (drives 2^n)
cp_counts = Counter(len(r.get("selected_conflict_pairs",[])) for r in results)
print("\nConflict pairs per example (n → 2^n extensions):")
for k in sorted(cp_counts):
    print(f"  {k} pair(s): {cp_counts[k]:3d} ({cp_counts[k]/n*100:.1f}%) → expect {2**k} ext")

# PRO kept
pro_counts = Counter(sum(1 for nid in r["nodes"] if nid.startswith("P")) for r in results)
print("\nPRO arguments kept per example:")
for k in sorted(pro_counts):
    print(f"  {k} PRO: {pro_counts[k]:3d} ({pro_counts[k]/n*100:.1f}%)")

# CON kept
con_counts = Counter(sum(1 for nid in r["nodes"] if nid.startswith("N")) for r in results)
print("\nCON arguments kept per example:")
for k in sorted(con_counts):
    print(f"  {k} CON: {con_counts[k]:3d} ({con_counts[k]/n*100:.1f}%)")

# edges
print(f"\nAvg supports (edges): {np.mean([len(r['supports']) for r in results]):.2f}")
print(f"Avg attacks (edges): {np.mean([len(r['effective_attacks']) for r in results]):.2f}")

# multi-extension
multi = sum(1 for r in results if len(r["preferred_extensions"]) > 1)
print(f"\nExamples with MULTIPLE preferred extensions: {multi} ({multi/n*100:.1f}%)")

# multi-extension accuracy split (the semantics finding)
multi_ex = [r for r in results if len(r["preferred_extensions"]) > 1]
single_ex = [r for r in results if len(r["preferred_extensions"]) == 1]
print(f"\n--- Accuracy split by extension count ---")
for name, subset in [("multi-extension", multi_ex), ("single-extension", single_ex)]:
    if subset:
        g = [r["gold"] for r in subset]
        line = f"  {name} ({len(subset)}): "
        for rule in ["credulous","skeptical","majority"]:
            p = [r["preferred_decisions"][rule] for r in subset]
            line += f"{rule}={accuracy_score(g,p)*100:.1f}%  "
        print(line)

=== αNLI — Extension Statistics (500 examples) ===

PREFERRED extensions per example:
  1 extension(s): 427 examples (85.4%)
  2 extension(s):  55 examples (11.0%)
  4 extension(s):  17 examples (3.4%)
  8 extension(s):   1 examples (0.2%)
  → avg: 1.23 | max: 8

STABLE extensions per example:
  1 extension(s): 427 examples (85.4%)
  2 extension(s):  55 examples (11.0%)
  4 extension(s):  17 examples (3.4%)
  8 extension(s):   1 examples (0.2%)
  → avg: 1.23

Conflict pairs per example (n → 2^n extensions):
  0 pair(s): 427 (85.4%) → expect 1 ext
  1 pair(s):  55 (11.0%) → expect 2 ext
  2 pair(s):  17 (3.4%) → expect 4 ext
  3 pair(s):   1 (0.2%) → expect 8 ext

PRO arguments kept per example:
  0 PRO:  18 (3.6%)
  1 PRO:  57 (11.4%)
  2 PRO:  77 (15.4%)
  3 PRO: 200 (40.0%)
  4 PRO: 148 (29.6%)

CON arguments kept per example:
  0 CON: 288 (57.6%)
  1 CON:  86 (17.2%)
  2 CON: 123 (24.6%)
  3 CON:   3 (0.6%)

Avg supports (edges): 2.79
Avg attacks (edges): 1.05

Examples with MULTIPL

## Decision rules recomputed with the final definitions

The run above used an earlier version of the decision rules (majority as "at least half", No-objection not yet implemented). This cell recomputes every rule from the saved frameworks, with no API calls, using the definitions of the paper (Section 3). It was run on the saved frameworks after the main run; the frameworks themselves are unchanged.

In [ ]:
# ===== RECOMPUTE DECISION RULES (no API calls) =====
# It reloads the saved frameworks and recomputes every decision rule with the
# definitions used in the paper:
#   majority     : claim in STRICTLY more than half of the extensions
#   No-objection : claim in at least one extension AND no attacker of the claim
#                  (direct or derived) in any extension
#   grounded     : least fixed point of the characteristic function
import pickle, glob
from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# load the saved frameworks (results/per_claim/deepseek_anli_500.pkl in the repository,
# or the checkpoint of the run above if it is still there)
ck = Path("/kaggle/working/anli_ext/anli_ckpt.pkl")
if ck.exists():
    d = pickle.load(open(ck, "rb")); results = [d[k] for k in sorted(d)]
else:
    results = pickle.load(open("../results/per_claim/deepseek_anli_500.pkl", "rb"))
print("frameworks loaded:", len(results))

def grounded(args, att):
    att = set(map(tuple, att)); g = set()
    while True:
        new = {a for a in args
               if all(any((y, x) in att for y in g) for (x, t) in att if t == a)}
        if new == g: return g
        g = new

def rules(r):
    exts = [set(e) for e in r["preferred_extensions"]]
    att = [tuple(e) for e in r["effective_attacks"]]
    cc = sum("CLAIM" in e for e in exts); m = len(exts)
    attackers = {s for s, t in att if t == "CLAIM"}
    in_any = set().union(*exts) if exts else set()
    return {"credulous": int(cc >= 1),
            "skeptical": int(m > 0 and cc == m),
            "majority": int(cc * 2 > m),
            "no_objection": int(cc >= 1 and not (attackers & in_any)),
            "grounded": int("CLAIM" in grounded(set(r["nodes"]), att))}

gold = [r["gold"] for r in results]
preds = {k: [] for k in ["credulous", "no_objection", "majority", "skeptical", "grounded"]}
for r in results:
    x = rules(r)
    for k in preds: preds[k].append(x[k])

rows = []
for k, p in preds.items():
    tn, fp, fn, tp = confusion_matrix(gold, p, labels=[0, 1]).ravel()
    rows.append({"rule": k, "acc": round(accuracy_score(gold, p)*100, 1),
                 "prec": round(precision_score(gold, p, zero_division=0)*100, 1),
                 "rec": round(recall_score(gold, p, zero_division=0)*100, 1),
                 "f1": round(f1_score(gold, p, zero_division=0)*100, 1),
                 "TN": tn, "FP": fp, "FN": fn, "TP": tp})
print(pd.DataFrame(rows).to_string(index=False))
same = all(preds["skeptical"] == preds[k] for k in ["majority", "no_objection", "grounded"])
print("\nskeptical = majority = No-objection = grounded on every claim:", same)


frameworks loaded: 500
        rule  acc  prec  rec   f1  TN  FP  FN  TP
   credulous 69.6  65.2 84.0 73.4 138 112  40 210
no_objection 71.4  68.5 79.2 73.5 159  91  52 198
    majority 71.4  68.5 79.2 73.5 159  91  52 198
   skeptical 71.4  68.5 79.2 73.5 159  91  52 198
    grounded 71.4  68.5 79.2 73.5 159  91  52 198

skeptical = majority = No-objection = grounded on every claim: True
